## Описание скрипта

In [1]:
# Импортируем библиотеку
import pandas as pd

### Определение файлов для обработки

**Что делаем:** 

Создаём словарь (тип данных dict), где:
- Ключ — год (число)
- Значение — путь к файлу (строка)

**Для чего:**

Храним все файлы в одном месте. Легко добавить новые годы (2022, 2023). В цикле `for year, filename in files.items()` будем перебирать все пары.

In [2]:
files = {
    2019: r"C:\...\2300_2019.csv",
    2020: r"C:\...\2300_2020.csv",
    2021: r"C:\...\2300_2021.csv"
}

### Порядок показатлей

**Что делаем:**

Создаём список строк — названия всех показателей в том порядке, в котором они должны идти в итоговом файле.

**Для чего:**

В исходных CSV файлах показатели идут в определённом порядке. В словаре `values_dict` ключи хранятся в произвольном порядке (так работает Python). Когда мы будем создавать итоговые строки, мы пройдём именно по этому списку, чтобы сохранить правильную последовательность.

In [3]:
indicator_order = [
    "Взято на учет рецидивов",
    "Взято на учет рецидивов III группы",
    "Прибыло",
    "Переведено в III группу",
    "Диагноз туберкулеза снят",
    "Выбыло",
    "Умерло от туберкулеза",
    "Умерло от других причин"
]

### Соответсвие названий показателей

**Что делаем:** Создаём словарь, который сопоставляет "сырые" названия из CSV с "эталонными" названиями.

**Для чего:**

В исходных файлах названия могут быть записаны по-разному: с лишними пробелами: " из них из Ш группы", без пробелов: "из них из Ш группы", с римской цифрой: "III" вместо "3". Словарь говорит: "если встретишь такую строку — замени её на такую".

**Структура словаря:**

- Ключ (что ищем в файле): " из них из Ш группы"
- Значение (что запишем в результат): "Взято на учет рецидивов III группы"

Несколько ключей ведут к одному значению, потому что один и тот же показатель может встречаться в разных форматах в разных файлах или даже в одном файле.

In [4]:
indicators_by_name = {
    "Взято на учет рецидивов": "Взято на учет рецидивов",
    "    из них из Ш группы": "Взято на учет рецидивов III группы",
    "из них из Ш группы": "Взято на учет рецидивов III группы",
    "Прибыло": "Прибыло",
    "Переведено в Ш группу": "Переведено в III группу",
    "Диагноз туберкулеза снят": "Диагноз туберкулеза снят",
    "Выбыло": "Выбыло",
    "Умерло от туберкулеза": "Умерло от туберкулеза",
    "Умерло от других причин": "Умерло от других причин"
}

### Возможные комбинации уточнений

**Что делаем:** Создаём список кортежей (пар) — все возможные сочетания "Уточнение" и "Еще уточнение".

**Для чего:**

В итоговом файле должно быть два уровня детализации:
- Уточнение: тип туберкулеза (органы дыхания, легкие, прочие формы)
- Еще уточнение: категория пациентов (всего, дети, подростки)

Всего получается 3 типа × 3 категории = 9 комбинаций

**Структура кортежа:** ("Уточнение", "Еще уточнение")

In [5]:
categories = [
    ("Туберкулез органов дыхания", "Всего"),
    ("Туберкулез органов дыхания", "Дети от 0 до 14 лет"),
    ("Туберкулез органов дыхания", "Подростки от 15 до 17 лет"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Всего"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Дети от 0 до 14 лет"),
    ("Туберкулез органов дыхания из них туберкулез легких", "Подростки от 15 до 17 лет"),
    ("Прочие формы туберкулеза", "Всего"),
    ("Прочие формы туберкулеза", "Дети от 0 до 14 лет"),
    ("Прочие формы туберкулеза", "Подростки от 15 до 17 лет")
]

### Соответствие комбинаций номерам колонок

**Что делаем:** Создаём словарь, где:

- Ключ — кортеж из двух строк (уточнение, ещё уточнение)
- Значение — номер колонки в исходном CSV, где лежат эти данные

**Для чего:**

- Исходный CSV файл не имеет заголовков (мы читаем с header=None)
- Вместо названий колонок используются номера: 0, 1, 2, 3, 4, 5, ...

Нам нужно знать, в какой колонке какое значение лежит

In [6]:
category_col_map = {
    ("Туберкулез органов дыхания", "Всего"): 4,
    ("Туберкулез органов дыхания", "Дети от 0 до 14 лет"): 5,
    ("Туберкулез органов дыхания", "Подростки от 15 до 17 лет"): 6,
    ("Туберкулез органов дыхания из них туберкулез легких", "Всего"): 7,
    ("Туберкулез органов дыхания из них туберкулез легких", "Дети от 0 до 14 лет"): 8,
    ("Туберкулез органов дыхания из них туберкулез легких", "Подростки от 15 до 17 лет"): 9,
    ("Прочие формы туберкулеза", "Всего"): 10,
    ("Прочие формы туберкулеза", "Дети от 0 до 14 лет"): 11,
    ("Прочие формы туберкулеза", "Подростки от 15 до 17 лет"): 12
}

### Подготовка к сбору данных
Создаём пустой список. Сюда мы будем добавлять словари (по одному на каждую строку итоговой таблицы). В конце список будет передан в pandas для создания DataFrame.

**Почему список словарей:** Это самый удобный способ накопить данные перед созданием DataFrame. Каждый словарь — одна строка, где ключи — названия колонок, значения — данные.

### Основной цикл по годам
**Что делаем:** Запускаем цикл по словарю `files.` На каждой итерации:
- `year` получает ключ (2019, 2020, 2021)
- `filename` получает значение (путь к файлу)

**Для чего:** Обработать каждый файл отдельно и добавить год ко всем строкам из этого файла.

Метод `.items()`: Возвращает пары (ключ, значение) из словаря.

### Чтение CSV файла
**Что делаем:** Читаем CSV файл в DataFrame.

**Параметры:**
- `filename` — путь к файлу
- `header=None` — говорит pandas, что в файле нет строки с заголовками. Все строки — это данные. Колонки будут называться 0, 1, 2, ...
- `encoding='utf-8'` — кодировка файла. Если файл в другой кодировке (например, cp1251), нужно изменить.

**Что получаем:** DataFrame df_raw, где:
- Строки соответствуют строкам файла
- Колонки имеют номера: df_raw[0], df_raw[1], ...

### Создание временного словаря
**Что делаем:** Создаём пустой словарь для текущего года.

**Для чего:**
- Этот словарь будет хранить все значения, найденные в текущем файле
- Ключ словаря: кортеж (показатель, уточнение, ещё_уточнение)
- Значение словаря: число (значение из таблицы)

После обработки всех строк файла, мы "развернём" этот словарь в строки итоговой таблицы

*Почему словарь, а не список:*

Словарь позволяет быстро проверить, есть ли уже значение для данной комбинации. Словарь не хранит дубликаты. К словарю легко обращаться по ключу: `values_dict.get(key, 0)`

### Внутренний цикл по строкам файла
**Что делаем:** Запускаем цикл по всем строкам `DataFrame df_raw`.

Метод `.iterrows()`: Возвращает пары (индекс_строки, содержимое_строки).
- `idx` — номер строки (0, 1, 2, ...)
- `row` — это Series (как строка таблицы), к колонкам можно обращаться по номеру: `row[0]`, `row[2]`

### Поиск названия показателя в строке
**Что делаем:** Пытаемся найти название показателя в строке.

**Логика:** Сначала проверяем колонку 0 (row[0]) --> Если в колонке 0 есть непустое значение — берём его --> Если колонка 0 пуста — проверяем колонку 2 (row[2])

**Проверки:**
- `pd.notna(row[0])` — проверяет, что значение не является NaN
- `str(row[0]).strip()` — преобразует в строку и удаляет пробелы по краям
- Если после `strip()` строка не пустая — условие истинно

*Почему проверяем и колонку 0, и колонку 2:*

- В файле названия показателей могут быть в разных местах
- Основное название — в колонке 0
- Для подстрок (например, "из них из Ш группы") название может быть в колонке 2

### Сопоставление с эталонными названиями
**Что делаем:** Ищем, соответствует ли найденное название одному из известных показателей.

**Логика:**
- Проходим по всем ключам словаря `indicators_by_name`
- Проверяем: содержится ли ключ в найденном названии (`key in indicator_name`)
- Или совпадает ли название полностью (`indicator_name == key`)
- Если да — берём эталонное название (`indicators_by_name[key]`) и выходим из цикла (`break`)

*Пример:* `indicator_name` = " из них из Ш группы" (с пробелами) --> Ключ " из них из Ш группы" содержится в этой строке → True --> Берём значение "Взято на учет рецидивов III группы".

*Почему `break`:* Как только нашли совпадение, дальше проверять не нужно — это ускоряет работу.

### Обработка найденного показателя
**Что делаем:** Проверяем, нашли ли мы совпадение. Если `matched_indicator` не None — значит, показатель опознан.

### Внутренний цикл по комбинациям уточнений
**Что делаем:** Проходим по всем парам (уточнение, ещё_уточнение) и соответствующим номерам колонок.

Метод `.items()` словаря `category_col_map` возвращает:

- Ключ: кортеж (`utochnenie`, `ese_utochnenie`)
- Значение: номер колонки `col_idx`

*Пример одной итерации:* 

- `utochnenie` = "Туберкулез органов дыхания"
- `ese_utochnenie` = "Всего"
- `col_idx` = 4

### Очистка и преобразование значения 
**Что делаем:** Проверяем, что значение валидное, и преобразуем его в целое число.

Пошагово:
- `pd.notna(value)` — значение не NaN?
- `str(value).strip()` — преобразуем в строку и убираем пробелы
- Проверяем, что результат не пустая строка и не строковые представления NaN
- Если всё ок — пробуем преобразовать в целое число:
    - `float(value)` — сначала в число с плавающей точкой (на случай "342.0")
    - `int(...)` — затем в целое число
- Если преобразование не удалось (`except`) — ставим 0
- Если значение пустое — ставим 0

*Почему через `float`:* В CSV числа могут быть записаны как "342" или "342.0". int("342.0") вызовет ошибку, а int(float("342.0")) сработает.

### Сохраняем в словарь
**Что делаем:** Сохраняем значение в словарь `values_dict`.

Ключ: Кортеж из трёх элементов — показатель, уточнение, ещё_уточнение.

Условие:
- Если такого ключа ещё нет в словаре — добавляем
- ИЛИ если значение по этому ключу равно 0 — заменяем (на случай, если сначала нашли 0, а потом реальное число)

*Почему такая проверка:* В файле показатель может встретиться дважды (например, в колонке 0 и колонке 2). Мы хотим сохранить первое ненулевое значение.

### Заполнение всех комбинаций после обработки файла
**Что делаем:** После того как весь файл прочитан и словарь `values_dict` заполнен, создаём строки для всех возможных комбинаций.

**Вложенные циклы:**
- Внешний цикл: по всем показателям в заданном порядке (`indicator_order`)
- Внутренний цикл: по всем комбинациям уточнений (`categories`)

**Метод .get():**
- Пытается получить значение из словаря по ключу
- Если ключа нет — возвращает значение по умолчанию (второй аргумент, здесь 0)

**Создание строки:**

all_data.append({
    'Показатель': indicator,
    'Уточнение': utochnenie,
    'Еще уточнение': ese_utochnenie,
    'Год': year,
    'Значение': value
})

Каждый такой словарь — одна строка итоговой таблицы.

*Почему 0 для отсутствующих значений:* В итоговой таблице должны быть все комбинации, даже если в исходных данных их нет. Это называется "плотная" (dense) таблица.

In [7]:
# Подготовка к сбору данных
all_data = []

# Основной цикл по годам
for year, filename in files.items():
    print(f"\nОбработка {year} года...")
    
    # Чтение CSV файла
    df_raw = pd.read_csv(filename, header=None, encoding='utf-8')
    
    # Создание временного словаря
    values_dict = {}
    
    # Внутренний цикл по строкам файла
    for idx, row in df_raw.iterrows():
        # Поиск названия показателя в строке
        indicator_name = ""
        if pd.notna(row[0]) and str(row[0]).strip():
            indicator_name = str(row[0]).strip()
        elif pd.notna(row[2]) and str(row[2]).strip():
            indicator_name = str(row[2]).strip()
        
        # Очищаем от лишних пробелов
        indicator_name = indicator_name.strip()
        
        # Сопоставление с эталонными названиями
        matched_indicator = None
        for key in indicators_by_name:
            if key in indicator_name or indicator_name == key:
                matched_indicator = indicators_by_name[key]
                break
        
        # Обработка найденного показателя
        if matched_indicator:
            print(f"  Найден показатель: '{matched_indicator}' (исходное: '{indicator_name}')")
            
            # Внутренний цикл по комбинациям уточнений
            for (utochnenie, ese_utochnenie), col_idx in category_col_map.items():
                #Проверка существования колонки. Проверяем, что номер колонки не превышает количество колонок в строке. Защита от ошибок — если файл вдруг имеет меньше колонок, чем ожидалось.
                if col_idx < len(row):
                    # Извлечение значения. Берём значение из нужной колонки текущей строки
                    value = row[col_idx]
                    
                    # Очистка и преобразование значения
                    if pd.notna(value) and str(value).strip() not in ['', 'nan', 'NaN', 'None']:
                        try:
                            value_int = int(float(value))
                        except:
                            value_int = 0
                    else:
                        value_int = 0
                    
                    # Сохраняем в словарь
                    key = (matched_indicator, utochnenie, ese_utochnenie)
                    if key not in values_dict or values_dict[key] == 0:
                        values_dict[key] = value_int
    
    # Заполнение всех комбинаций после обработки файла
    for indicator in indicator_order:
        for utochnenie, ese_utochnenie in categories:
            value = values_dict.get((indicator, utochnenie, ese_utochnenie), 0)
            
            all_data.append({
                'Показатель': indicator,
                'Уточнение': utochnenie,
                'Еще уточнение': ese_utochnenie,
                'Год': year,
                'Значение': value
            })


Обработка 2019 года...
  Найден показатель: 'Взято на учет рецидивов' (исходное: 'Взято на учет рецидивов')
  Найден показатель: 'Взято на учет рецидивов III группы' (исходное: 'из них из Ш группы')
  Найден показатель: 'Прибыло' (исходное: 'Прибыло')
  Найден показатель: 'Переведено в III группу' (исходное: 'Переведено в Ш группу')
  Найден показатель: 'Диагноз туберкулеза снят' (исходное: 'Диагноз туберкулеза снят')
  Найден показатель: 'Выбыло' (исходное: 'Выбыло')
  Найден показатель: 'Умерло от туберкулеза' (исходное: 'Умерло от туберкулеза')
  Найден показатель: 'Умерло от других причин' (исходное: 'Умерло от других причин')

Обработка 2020 года...
  Найден показатель: 'Взято на учет рецидивов' (исходное: 'Взято на учет рецидивов')
  Найден показатель: 'Взято на учет рецидивов III группы' (исходное: 'из них из Ш группы')
  Найден показатель: 'Прибыло' (исходное: 'Прибыло')
  Найден показатель: 'Переведено в III группу' (исходное: 'Переведено в Ш группу')
  Найден показатель: 'Ди

### Создаем DataFrame
**Что делаем:** Преобразуем список словарей в DataFrame pandas.

**Результат:** Таблица с колонками, которые были ключами в словарях: Показатель, Уточнение, Еще уточнение, Год, Значение.

In [8]:
result = pd.DataFrame(all_data)

### Сохранение результатов 
**Что делаем:** Сохраняем результат в два файла — Excel и CSV.

Параметр `index=False`: Не сохранять номера строк из pandas (лишний столбец с 0, 1, 2...).

Excel (to_excel): Сохраняет в формате .xlsx - Удобно для просмотра в Excel

CSV (to_csv): Сохраняет в текстовом формате с разделителями; `encoding='utf-8-sig'` — кодировка `UTF-8` с BOM (Excel правильно откроет русские буквы); sep=',' — разделитель — запятая.

In [9]:
# Сохраняем
output_excel = r"C:\...\результат_полный1.xlsx"
result.to_excel(output_excel, index=False)

output_csv = r"C:\...\результат_полный1.csv"
result.to_csv(output_csv, index=False, encoding='utf-8-sig', sep=',')